# 2026년 네팔 홍수: Leafmap × 바이브 코딩 (학생용)
### 실습 20~25분 | Google Colab
**질문:** 홍수 전후 호수 주변의 수역이 얼마나 달라졌을까요?

실습 목표: (1) 공식 폐색호 경계와 Sentinel-2 영상 보기, (2) MNDWI로 수역 추출, (3) 새로 탐지된 수역 면적 계산, (4) AI로 분류 기준 바꿔보기.

**사전 준비:** 교수자가 제공한 `sentinel_before.tif`, `sentinel_after.tif` **2개**가 필요합니다. 두 파일은 동일한 영역·좌표계·20m 격자로 정렬된 **6밴드 GeoTIFF**이며 순서는 `Red, Green, Blue, NIR, SWIR1, SCL`입니다. 앞의 네 밴드는 반사도(float32)로 보정되어 있어야 합니다. **영상 다운로드는 수업에서 하지 않습니다.**

> 이 실습은 홍수 피해 면적의 확정 산정이 아니라 **AI 코딩 및 수역 변화 탐지 체험**입니다. 구름·눈·산 그림자와 촬영일 차이 때문에 오분류될 수 있습니다.

## 수업 순서
| 시간 | 학생 활동 |
|---|---|
| 0–4분 | 모듈 준비 + 위성자료 업로드 |
| 4–8분 | Leafmap으로 공식 호수 경계 확인 |
| 8–13분 | 전후 위성영상과 MNDWI 비교 |
| 13–19분 | 수역 변화량 계산 및 지도 표현 |
| 19–25분 | AI에게 임계값 변경 요청 후 결과 해석 |

교수자가 영상 파일을 Drive에 배포했다면 업로드 대신 파일 경로만 변경하세요.

## 1. 모듈 설치

In [ ]:
%pip -q install leafmap geopandas rasterio folium matplotlib
!apt-get -qq install fonts-nanum

## 2. 공식 GeoJSON 자료 읽기
호수 경계는 위성영상 수역 추출의 **참고자료**입니다. 연구지역 전체 피해 범위와는 다릅니다.

In [ ]:
import geopandas as gpd
import leafmap.foliumap as leafmap
import folium
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.transform import array_bounds
from rasterio.features import geometry_mask

base_url = "https://raw.githubusercontent.com/KU-EBL/Data/main/GIS"
lake = gpd.read_file(f"{base_url}/Barrier_lake.geojson").to_crs("EPSG:4326")
flood = gpd.read_file(f"{base_url}/Flood_extent.geojson").to_crs("EPSG:4326")
print(f"호수 객체 {len(lake)}개 / 홍수 참고 객체 {len(flood)}개")
print("호수 경계:", lake.total_bounds)

m = leafmap.Map(center=[28.31, 85.50], zoom=12, height=460, tiles=None)
m.add_basemap("SATELLITE")
m.add_gdf(flood, layer_name="공식 영향 범위(참고)",
          style={"color":"#00BFFF","weight":1,"fillColor":"#00BFFF","fillOpacity":0.12})
m.add_gdf(lake, layer_name="UNOSAT 폐색호",
          style={"color":"red","weight":3,"fillOpacity":0.1})
m.zoom_to_gdf(lake)
m

## 3. 교수자가 제공한 전후 위성영상 다운로드
**두 TIFF를 동시에 선택**하세요. 이미 `/content/`에 파일이 있다면 업로드 줄을 건너뛰어도 됩니다.

주의: 제공 영상은 반사도 보정 및 격자 정렬이 완료된 수업용 파일을 전제로 합니다.

In [ ]:
import requests
import rasterio
import numpy as np
from pathlib import Path

base_url = "https://raw.githubusercontent.com/KU-EBL/Data/main/GIS"

#다운로드할 위성영상
satellite_files = [
    "sentinel_before.tif",
    "sentinel_after.tif"
]

#Colab 저장 폴더
output_dir = Path("/content/nepal_sentinel")
output_dir.mkdir(parents=True, exist_ok=True)

# 4. GitHub에서 영상 다운로드
for filename in satellite_files:

    url = f"{base_url}/{filename}"
    save_path = output_dir / filename

    print(f"다운로드 중: {filename}")

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    save_path.write_bytes(response.content)

    print(f"다운로드 완료: {filename}")

#영상 경로
before_path = output_dir / "sentinel_before.tif"
after_path = output_dir / "sentinel_after.tif"

#영상 정보 및 격자 확인
with rasterio.open(before_path) as b, rasterio.open(after_path) as a:

    assert b.count >= 5 and a.count >= 5, \
        "5밴드 Red/Green/NIR/SWIR1/SCL 자료가 필요합니다."

    assert (
        b.crs == a.crs
        and b.transform == a.transform
        and b.shape == a.shape
    ), "전후 영상 격자가 다릅니다."

    assert b.crs.is_projected, \
        "면적 계산을 위해 투영좌표계가 필요합니다."

    # 영상 읽기
    before_data = b.read().astype("float32")
    after_data = a.read().astype("float32")

    # 공간정보
    transform = b.transform
    crs = b.crs
    bounds0 = b.bounds

    # 픽셀 면적
    pixel_area = abs(
        transform.a * transform.e -
        transform.b * transform.d
    )

    print("\n영상 정보")
    print("영상 크기:", b.shape)
    print("좌표계:", crs)
    print("픽셀 면적(m²):", pixel_area)

print("\n밴드 순서: Red / Green / NIR / SWIR1 / SCL")
print("홍수 전후 영상 불러오기 완료!")

## 4. 다운로드한 위성 영상을 베이스 맵에서 확인하기

In [ ]:
import numpy as np
import folium
import leafmap.foliumap as leafmap

from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling
)
from rasterio.transform import array_bounds


# ==================================================
# 1. Sentinel-2 자연색 RGB 영상 설정
# ==================================================

# TIFF 밴드 순서
# 0: Red (B04)
# 1: Green (B03)
# 2: NIR (B08)
# 3: SWIR1 (B11)
# 4: SCL
# 5: Blue (B02)

assert before_data.shape[0] >= 6
assert after_data.shape[0] >= 6, \
    "Blue 밴드가 포함된 6밴드 영상이 필요합니다."


# 유효 픽셀 마스크
def get_valid_mask(data):

    red = data[0]
    green = data[1]
    blue = data[5]
    scl = data[4]

    return (
        np.isfinite(red) &
        np.isfinite(green) &
        np.isfinite(blue) &
        (red >= 0) &
        (green >= 0) &
        (blue >= 0) &
        np.isin(scl.astype(int), [2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
    )


before_valid = get_valid_mask(before_data)
after_valid = get_valid_mask(after_data)


# ==================================================
# 2. 홍수 전후 공통 RGB 밝기 조정
# ==================================================

# 자연색 RGB: Red, Green, Blue
rgb_band_indices = [0, 1, 5]

stretch_limits = []

for band_index in rgb_band_indices:

    values = np.concatenate([
        before_data[band_index][before_valid],
        after_data[band_index][after_valid]
    ])

    values = values[np.isfinite(values)]

    if values.size == 0:
        raise ValueError("유효한 영상 픽셀이 없습니다.")

    # 2~98% Percentile Stretch
    p2, p98 = np.percentile(values, [2, 98])

    stretch_limits.append((p2, p98))

print("RGB 밝기 조정 범위:", stretch_limits)


# ==================================================
# 3. Sentinel-2 자연색 영상 생성 및 좌표 변환
# ==================================================

def create_sentinel_overlay(data, crs, transform, bounds0):

    # Red, Green, Blue
    red = data[0]
    green = data[1]
    blue = data[5]

    valid = get_valid_mask(data)

    bands = [red, green, blue]

    rgb_channels = []

    for band, (p2, p98) in zip(bands, stretch_limits):

        # Percentile Stretch
        scaled = np.clip(
            (band - p2) / max(p98 - p2, 1e-6),
            0,
            1
        )

        # Gamma Correction: 어두운 영역 밝게 표시
        gamma = 0.55
        scaled = scaled ** gamma

        rgb_channels.append(
            (scaled * 255).astype("uint8")
        )

    # RGB 영상 생성
    rgb = np.stack(rgb_channels, axis=-1)

    # NoData 픽셀 투명 처리
    alpha = np.where(
        valid, 255, 0
    ).astype("uint8")

    rgba = np.dstack([rgb, alpha])

    # EPSG:4326 좌표계로 변환
    height, width = red.shape

    dst_transform, dst_width, dst_height = (
        calculate_default_transform(
            crs,
            "EPSG:4326",
            width,
            height,
            *bounds0
        )
    )

    dst = np.zeros(
        (dst_height, dst_width, 4),
        dtype="uint8"
    )

    for i in range(4):

        reproject(
            source=rgba[:, :, i],
            destination=dst[:, :, i],
            src_transform=transform,
            src_crs=crs,
            dst_transform=dst_transform,
            dst_crs="EPSG:4326",
            src_nodata=0 if i == 3 else None,
            dst_nodata=0,
            resampling=Resampling.nearest
        )

    west, south, east, north = array_bounds(
        dst_height,
        dst_width,
        dst_transform
    )

    bounds = [
        [south, west],
        [north, east]
    ]

    return dst, bounds


# ==================================================
# 4. 홍수 전후 자연색 RGB 영상 변환
# ==================================================

before_img, before_bounds = create_sentinel_overlay(
    before_data,
    crs,
    transform,
    bounds0
)

after_img, after_bounds = create_sentinel_overlay(
    after_data,
    crs,
    transform,
    bounds0
)

print("Sentinel-2 자연색 RGB 영상 변환 완료")


# ==================================================
# 5. Leafmap 지도 생성
# ==================================================

m_sentinel = leafmap.Map(
    center=[28.31, 85.50],
    zoom=12,
    height=650,
    tiles=None
)

# 위성 배경지도
m_sentinel.add_basemap("SATELLITE")


# ==================================================
# 6. 홍수 전후 Sentinel-2 자연색 영상 중첩
# ==================================================

# 홍수 이전 영상
folium.raster_layers.ImageOverlay(
    image=before_img,
    bounds=before_bounds,
    name="Before Flood - RGB (2026.08.12)",
    opacity=1.0,
    show=True
).add_to(m_sentinel)

# 홍수 이후 영상
folium.raster_layers.ImageOverlay(
    image=after_img,
    bounds=after_bounds,
    name="After Flood - RGB (2026.10.01)",
    opacity=1.0,
    show=False
).add_to(m_sentinel)


# ==================================================
# 7. UNOSAT 폐색호 경계 추가
# ==================================================

m_sentinel.add_gdf(
    lake.to_crs("EPSG:4326"),
    layer_name="UNOSAT Barrier Lakes",
    style={
        "color": "#FF3333",
        "weight": 3,
        "fillColor": "#FF3333",
        "fillOpacity": 0.05
    }
)


# ==================================================
# 8. 지도 확대 및 레이어 선택
# ==================================================

# 호수 위치를 중심으로 확대
m_sentinel.zoom_to_gdf(
    lake.to_crs("EPSG:4326")
)

# 레이어 선택 기능
folium.LayerControl(
    collapsed=False
).add_to(m_sentinel)

# 지도 출력
m_sentinel

## 5. 영상과 MNDWI 비교
MNDWI = (Green − SWIR1) / (Green + SWIR1). 값이 높을수록 수역일 가능성이 높습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================================
# 1. MNDWI 계산
# ==============================================

def get_mndwi(data):

    # 밴드: Red, Green, NIR, SWIR1, SCL, Blue
    red, green, nir, swir, scl = data[:5]

    # SCL 4: 식생, 5: 나지, 6: 수역
    valid = np.isin(scl, [4, 5, 6])

    # NoData 및 비정상 값 제외
    valid &= np.isfinite(green) & np.isfinite(swir)
    valid &= (green > 0) & (swir > 0)
    valid &= (green + swir > 1e-6)

    mndwi = np.full(
        green.shape,
        np.nan,
        dtype="float32"
    )

    mndwi[valid] = (
        (green[valid] - swir[valid]) /
        (green[valid] + swir[valid])
    )

    return mndwi, valid


# ==============================================
# 2. 홍수 전후 MNDWI 계산
# ==============================================

before_mndwi, before_valid = get_mndwi(before_data)
after_mndwi, after_valid = get_mndwi(after_data)

# 동일 위치에서 관측 가능한 픽셀만 비교
common_valid = before_valid & after_valid

print(
    f"공통 관측 가능 픽셀: "
    f"{common_valid.mean() * 100:.1f}%"
)


# ==============================================
# 3. MNDWI 변화량 계산
# ==============================================

mndwi_change = np.full(
    before_mndwi.shape,
    np.nan,
    dtype="float32"
)

mndwi_change[common_valid] = (
    after_mndwi[common_valid]
    - before_mndwi[common_valid]
)


# ==============================================
# 4. MNDWI 전후 및 변화량 시각화
# ==============================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5),
    constrained_layout=True
)

# Before
im1 = axes[0].imshow(
    np.where(common_valid, before_mndwi, np.nan),
    cmap="BrBG",
    vmin=-1,
    vmax=1
)

axes[0].set_title("Before Flood - MNDWI")
axes[0].axis("off")

# After
im2 = axes[1].imshow(
    np.where(common_valid, after_mndwi, np.nan),
    cmap="BrBG",
    vmin=-1,
    vmax=1
)

axes[1].set_title("After Flood - MNDWI")
axes[1].axis("off")

# Change
im3 = axes[2].imshow(
    mndwi_change,
    cmap="RdBu",
    vmin=-0.5,
    vmax=0.5
)

axes[2].set_title("MNDWI Change (After - Before)")
axes[2].axis("off")

fig.colorbar(
    im2,
    ax=axes[:2],
    shrink=0.7,
    label="MNDWI"
)

fig.colorbar(
    im3,
    ax=axes[2],
    shrink=0.7,
    label="ΔMNDWI"
)

plt.show()

## 6. 수역 변화량 계산
임계값 `threshold=0.0`은 연습용 시작값입니다. 결과를 확정 홍수 면적으로 해석하지 마세요.

In [ ]:
# ==============================================
# 1. MNDWI를 이용한 수역 분류
# ==============================================

threshold = 0.0

water_before = (
    (before_mndwi > threshold) & common_valid
)

water_after = (
    (after_mndwi > threshold) & common_valid
)

# 신규 수역 및 사라진 수역
new_water = water_after & ~water_before
receded_water = water_before & ~water_after


# ==============================================
# 2. UNOSAT 폐색호 주변 300m 분석 영역
# ==============================================

lake_local = lake.to_crs(crs)

# 호수 폴리곤을 합친 후 300m 버퍼
lake_zone = (
    lake_local.geometry.union_all().buffer(300)
)

# 분석 영역을 래스터 마스크로 변환
lake_mask = geometry_mask(
    [lake_zone],
    out_shape=water_before.shape,
    transform=transform,
    invert=True
)

# 두 날짜 모두 유효한 영역
mask = common_valid & lake_mask


# ==============================================
# 3. 수역 면적 계산 (ha)
# ==============================================

before_ha = (
    (water_before & mask).sum()
    * pixel_area / 10000
)

after_ha = (
    (water_after & mask).sum()
    * pixel_area / 10000
)

new_ha = (
    (new_water & mask).sum()
    * pixel_area / 10000
)

receded_ha = (
    (receded_water & mask).sum()
    * pixel_area / 10000
)

net_change = after_ha - before_ha


# ==============================================
# 4. 결과표
# ==============================================

summary = pd.DataFrame({
    "분석 지표": [
        "홍수 이전 수역",
        "홍수 이후 수역",
        "신규 수역 후보",
        "사라진 수역 후보",
        "순변화"
    ],
    "면적(ha)": [
        before_ha,
        after_ha,
        new_ha,
        receded_ha,
        net_change
    ]
})

display(summary.round(2))


# ==============================================
# 5. 관측 가능 영역 확인
# ==============================================

coverage = (
    100 * mask.sum() / lake_mask.sum()
    if lake_mask.sum() > 0 else np.nan
)

print(f"호수 주변 공통 관측 비율: {coverage:.1f}%")

print(f"수역 면적 순변화: {net_change:+.2f} ha")

# 계산 결과 검증
assert np.isclose(
    net_change,
    new_ha - receded_ha
), "수역 변화량 계산을 확인하세요."

## 7. Leafmap 위성지도에 변화만 중첩하기
**파랑:** 새로 수역으로 분류된 픽셀 / **주황:** 수역에서 제외된 픽셀 / **빨강 경계:** UNOSAT 호수

In [ ]:
import numpy as np
import geopandas as gpd
import folium
import leafmap.foliumap as leafmap

from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling
)
from rasterio.transform import array_bounds


# ==============================================
# 1. 수역 변화 분류
# ==============================================

# 0: 변화 없음 또는 분석 제외
# 1: 신규 수역
# 2: 사라진 수역

change = np.zeros(
    water_before.shape,
    dtype="uint8"
)

change[new_water & mask] = 1
change[receded_water & mask] = 2


# ==============================================
# 2. EPSG:4326으로 재투영
# ==============================================

wgs_transform, w, h = calculate_default_transform(
    crs,
    "EPSG:4326",
    change.shape[1],
    change.shape[0],
    *bounds0
)

dst = np.zeros((h, w), dtype="uint8")

reproject(
    source=change,
    destination=dst,
    src_transform=transform,
    src_crs=crs,
    dst_transform=wgs_transform,
    dst_crs="EPSG:4326",
    resampling=Resampling.nearest
)


# ==============================================
# 3. 변화지역 색상 지정
# ==============================================

rgba = np.zeros((h, w, 4), dtype="uint8")

# 신규 수역: 파란색
rgba[dst == 1] = [20, 120, 240, 230]

# 사라진 수역: 주황색
rgba[dst == 2] = [245, 140, 40, 230]

west, south, east, north = array_bounds(
    h, w, wgs_transform
)

map_bounds = [
    [south, west],
    [north, east]
]


# ==============================================
# 4. Leafmap 지도 생성
# ==============================================

m_change = leafmap.Map(
    center=[
        (south + north) / 2,
        (west + east) / 2
    ],
    zoom=12,
    height=600,
    tiles=None
)

m_change.add_basemap("SATELLITE")


# ==============================================
# 5. 수역 변화 결과 추가
# ==============================================

folium.raster_layers.ImageOverlay(
    image=rgba,
    bounds=map_bounds,
    name="수역 변화 (신규/감소)",
    opacity=0.9,
    show=True
).add_to(m_change)


# ==============================================
# 6. UNOSAT 폐색호 경계
# ==============================================

m_change.add_gdf(
    lake.to_crs("EPSG:4326"),
    layer_name="UNOSAT 폐색호",
    style={
        "color": "#FF3333",
        "weight": 3,
        "fillOpacity": 0
    }
)


# ==============================================
# 7. 호수 주변 300m 분석 범위
# ==============================================

lake_buffer_gdf = gpd.GeoDataFrame(
    geometry=[lake_zone],
    crs=crs
).to_crs("EPSG:4326")

m_change.add_gdf(
    lake_buffer_gdf,
    layer_name="분석 범위 (300m)",
    style={
        "color": "#FFFF00",
        "weight": 2,
        "fillOpacity": 0,
        "dashArray": "5, 5"
    }
)


# ==============================================
# 8. 범례 추가
# ==============================================

legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background: white;
    padding: 12px;
    border: 1px solid gray;
    border-radius: 6px;
    font-size: 13px;
">
<b>Water Area Change</b>
<p><span style="color:#1478F0;">■</span> 신규 수역 후보</p>
<p><span style="color:#F58C28;">■</span> 사라진 수역 후보</p>
<p><span style="color:#FF3333;">━</span> UNOSAT 폐색호</p>
<p><span style="color:#B5A000;">┄</span> 300m 분석 범위</p>
</div>
"""

m_change.get_root().html.add_child(
    folium.Element(legend_html)
)


# ==============================================
# 9. 지도 확대 및 출력
# ==============================================

# 분석 대상 호수 중심으로 확대
m_change.zoom_to_gdf(
    lake.to_crs("EPSG:4326")
)

folium.LayerControl(
    collapsed=False
).add_to(m_change)

m_change

## 8. 바이브 코딩(Vibe Coding): AI를 이용한 수역 변화 분석 (5분)

앞에서 실행한 코드와 변수를 활용하여, ChatGPT에 자연어로 분석을 요청해 봅시다.

**실습 방법**

아래 프롬프트를 ChatGPT에 입력합니다.

AI가 생성한 Python 코드를 Colab의 새로운 코드 셀에 붙여넣고 실행합니다.

오류가 발생하면 오류 메시지를 ChatGPT에 제공하여 수정합니다.

결과표와 그래프를 확인합니다.

**ChatGPT 프롬프트:**

"Google Colab에서 2026년 네팔 홍수 전후 Sentinel-2 영상을 분석하고 있습니다.

이미 before_mndwi, after_mndwi, common_valid, lake_mask, pixel_area 변수가 생성되어 있습니다. MNDWI는 2차원 NumPy 배열이고, 두 마스크는 Boolean 배열이며, pixel_area의 단위는 m²입니다.

다음 분석을 수행하는 Python 코드를 작성해 주세요.

MNDWI 임계값을 -0.1, 0.0, 0.1, 0.2로 변경합니다.

각 임계값에서 새롭게 수역으로 분류된 면적을 ha 단위로 계산합니다.

공통 유효 픽셀과 폐색호 주변 300m 영역만 분석합니다.

임계값별 신규 수역 후보 면적을 표와 막대그래프로 표시합니다.

한글 주석을 작성하고 Colab의 단일 코드 셀에서 실행되도록 해주세요."

**결과 확인 질문:**

임계값에 따라 신규 수역 후보 면적은 어떻게 변화하는가?

임계값에 따라 면적이 달라지는 이유는 무엇인가?

In [ ]:
# 학생 실습: 여기에 AI가 생성한 코드를 붙여넣으세요.
# 코드가 잘 안 되면 변수 이름과 mask = common_valid & lake_mask 사용 여부를 확인하세요.


## 부록: 위성영상 다운로드 코드

In [ ]:
# =====================================================
# 2026 네팔 홍수 전후 Sentinel-2 영상 다운로드
# Before: 2026-08-12
# After : 2026-10-01
#
# 밴드 순서:
# Red / Green / NIR / SWIR1 / SCL / Blue
#
# 출력: 20m, EPSG:32645, GeoTIFF
# =====================================================

%pip -q install pystac-client rasterio

import numpy as np
import rasterio
import pandas as pd

from pathlib import Path
from pystac_client import Client
from shapely.geometry import box, shape
from rasterio.warp import (
    transform_bounds,
    reproject,
    Resampling
)
from rasterio.transform import from_origin


# -----------------------------------------------------
# 1. 위성영상 검색 설정
# -----------------------------------------------------

catalog = Client.open(
    "https://earth-search.aws.element84.com/v1"
)

# 기존에 설정한 bbox 사용
# bbox = [min_lon, min_lat, max_lon, max_lat]

dates = {
    "before": "2026-08-12",
    "after": "2026-10-01"
}

bands = [
    "red",       # 1. B04
    "green",     # 2. B03
    "nir",       # 3. B08
    "swir16",    # 4. B11
    "scl",       # 5. Scene Classification
    "blue"       # 6. B02 (추가)
]


# -----------------------------------------------------
# 2. 지정한 날짜의 Sentinel-2 영상 검색
# -----------------------------------------------------

def search_sentinel(date):

    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{date}T00:00:00Z/{date}T23:59:59Z",
        max_items=100
    )

    items = list(search.items())

    # 필요한 밴드가 모두 존재하는 영상
    items = [
        item for item in items
        if all(band in item.assets for band in bands)
    ]

    if not items:
        raise ValueError(
            f"{date} 날짜의 Sentinel-2 영상이 없습니다."
        )

    # 연구지역과 실제로 겹치는 면적이 큰 영상 우선
    # 동일하면 전체 구름량이 적은 영상 선택
    target = box(*bbox)

    def ranking(item):
        overlap = shape(item.geometry).intersection(target).area
        cloud = item.properties.get("eo:cloud_cover")
        cloud = float(cloud) if cloud is not None else 100
        return (-overlap, cloud)

    items.sort(key=ranking)

    selected = items[0]

    print(f"\n선택 날짜: {date}")
    print("영상 ID:", selected.id)
    print("구름 비율:",
          selected.properties.get("eo:cloud_cover"), "%")

    return selected


before = search_sentinel(dates["before"])
after = search_sentinel(dates["after"])


# -----------------------------------------------------
# 3. 저장 위치 및 좌표계 설정
# -----------------------------------------------------

output_dir = Path("/content/nepal_sentinel")
output_dir.mkdir(parents=True, exist_ok=True)

target_crs = "EPSG:32645"
resolution = 20

xmin, ymin, xmax, ymax = transform_bounds(
    "EPSG:4326",
    target_crs,
    *bbox
)

# 동일한 픽셀 격자 사용
xmin = np.floor(xmin / resolution) * resolution
ymin = np.floor(ymin / resolution) * resolution
xmax = np.ceil(xmax / resolution) * resolution
ymax = np.ceil(ymax / resolution) * resolution

width = int(round((xmax - xmin) / resolution))
height = int(round((ymax - ymin) / resolution))

transform = from_origin(
    xmin,
    ymax,
    resolution,
    resolution
)


# -----------------------------------------------------
# 4. 위성영상 다운로드 함수
# -----------------------------------------------------

def download_sentinel(item, filename):

    path = output_dir / filename

    # Earth Search 자료의 BOA offset 적용 여부
    offset_applied = item.properties.get(
        "earthsearch:boa_offset_applied"
    )

    if offset_applied is None:
        raise ValueError(
            "반사도 오프셋 적용 여부를 확인할 수 없습니다. "
            "STAC 메타데이터 확인이 필요합니다."
        )

    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=len(bands),
        dtype="float32",
        crs=target_crs,
        transform=transform,
        nodata=-9999,
        compress="deflate"
    ) as dst:

        for i, band in enumerate(bands, start=1):

            print(f"  다운로드: {band}")

            asset = item.assets[band]

            with rasterio.open(asset.href) as src:

                data = np.full(
                    (height, width),
                    -9999,
                    dtype="float32"
                )

                reproject(
                    source=rasterio.band(src, 1),
                    destination=data,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    dst_nodata=-9999,
                    resampling=(
                        Resampling.nearest
                        if band == "scl"
                        else Resampling.bilinear
                    )
                )

            # 반사도 변환 (SCL은 분류값이므로 제외)
            if band != "scl":

                raster_info = asset.extra_fields.get(
                    "raster:bands", [{}]
                )

                scale = raster_info[0].get(
                    "scale", 0.0001
                )

                valid = data != -9999
                valid &= data > 0

                # Earth Search에서 아직 BOA 오프셋이
                # 적용되지 않은 자료에만 보정
                offset = 0 if offset_applied else -1000

                data[valid] = (
                    data[valid] + offset
                ) * float(scale)

                data[~valid] = -9999

            dst.write(data, i)
            dst.set_band_description(i, band)

        dst.update_tags(
            acquisition_date=str(item.datetime.date()),
            source_item=item.id,
            radiometry="surface_reflectance_already_scaled",
            band_order="red,green,nir,swir16,scl,blue"
        )

    print("저장 완료:", path)
    return path


# -----------------------------------------------------
# 5. 홍수 전후 영상 다운로드
# -----------------------------------------------------

before_path = download_sentinel(
    before,
    "sentinel_before.tif"
)

after_path = download_sentinel(
    after,
    "sentinel_after.tif"
)


# -----------------------------------------------------
# 6. 다운로드 영상 확인
# -----------------------------------------------------

for path in [before_path, after_path]:

    with rasterio.open(path) as src:

        print("\n파일:", path.name)
        print("촬영일:", src.tags().get("acquisition_date"))
        print("크기:", src.width, "x", src.height)
        print("좌표계:", src.crs)
        print("해상도:", src.res)
        print("밴드 수:", src.count)
        print("밴드:", src.descriptions)

# 전후 격자 일치 확인
with rasterio.open(before_path) as b, \
     rasterio.open(after_path) as a:

    assert b.crs == a.crs
    assert b.transform == a.transform
    assert b.shape == a.shape
    assert b.count == a.count == 6

    print("\n전후 영상 격자 일치 확인 완료")

print("\nSentinel-2 다운로드 완료!")

In [ ]:
from google.colab import files

files.download(str(before_path))
files.download(str(after_path))